<a href="https://colab.research.google.com/github/Shahul187/aml-alert-triage/blob/main/notebooks/01_data_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

folder = '/content/drive/MyDrive/aml-project'

for f in os.listdir(folder):
    size_mb = os.path.getsize(os.path.join(folder, f)) / 1e6
    print(f"{f}  ({size_mb:.1f} MB)")

SAML-D.csv  (996.2 MB)


In [3]:
import pandas as pd

file = '/content/drive/MyDrive/aml-project/SAML-D.csv'

peek = pd.read_csv(file, nrows=1000)

print("Shape of peek:", peek.shape)
print("\nColumns:")
for col in peek.columns:
    print(f"  {col:<25} {peek[col].dtype}")

Shape of peek: (1000, 12)

Columns:
  Time                      object
  Date                      object
  Sender_account            int64
  Receiver_account          int64
  Amount                    float64
  Payment_currency          object
  Received_currency         object
  Sender_bank_location      object
  Receiver_bank_location    object
  Payment_type              object
  Is_laundering             int64
  Laundering_type           object


In [4]:
with open(file) as f:
    total_rows = sum(1 for _ in f) - 1

print(f"Total rows: {total_rows:,}")

Total rows: 9,504,852


In [5]:
counts = pd.read_csv(file, usecols=['Is_laundering'])['Is_laundering'].value_counts()

clean, criminal = counts[0], counts[1]
total = clean + criminal

print(f"Clean:     {clean:>10,}  ({clean/total:.4%})")
print(f"Laundering:{criminal:>10,}  ({criminal/total:.4%})")
print(f"\nRatio: 1 in {round(total/criminal):,} transactions")

Clean:      9,494,979  (99.8961%)
Laundering:     9,873  (0.1039%)

Ratio: 1 in 963 transactions


In [6]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print(peek.head(5))
print("\n--- Amount summary ---")
print(peek['Amount'].describe())

       Time        Date  Sender_account  Receiver_account    Amount Payment_currency Received_currency Sender_bank_location Receiver_bank_location  Payment_type  Is_laundering       Laundering_type
0  10:35:19  2022-10-07      8724731955        2769355426   1459.15        UK pounds         UK pounds                   UK                     UK  Cash Deposit              0  Normal_Cash_Deposits
1  10:35:20  2022-10-07      1491989064        8401255335   6019.64        UK pounds            Dirham                   UK                    UAE  Cross-border              0        Normal_Fan_Out
2  10:35:20  2022-10-07       287305149        4404767002  14328.44        UK pounds         UK pounds                   UK                     UK        Cheque              0  Normal_Small_Fan_Out
3  10:35:21  2022-10-07      5376652437        9600420220  11895.00        UK pounds         UK pounds                   UK                     UK           ACH              0         Normal_Fan_In
4  10:35:2

In [7]:
lt = pd.read_csv(file, usecols=['Laundering_type', 'Is_laundering'])

print("=== CRIMINAL patterns ===")
print(lt[lt['Is_laundering'] == 1]['Laundering_type'].value_counts())

print("\n=== NORMAL patterns ===")
print(lt[lt['Is_laundering'] == 0]['Laundering_type'].value_counts())

=== CRIMINAL patterns ===
Laundering_type
Structuring             1870
Cash_Withdrawal         1334
Deposit-Send             945
Smurfing                 932
Layered_Fan_In           656
Layered_Fan_Out          529
Stacked Bipartite        506
Behavioural_Change_1     394
Bipartite                383
Cycle                    382
Fan_In                   364
Gather-Scatter           354
Behavioural_Change_2     345
Scatter-Gather           338
Single_large             250
Fan_Out                  237
Over-Invoicing            54
Name: count, dtype: int64

=== NORMAL patterns ===
Laundering_type
Normal_Small_Fan_Out      3477717
Normal_Fan_Out            2302220
Normal_Fan_In             2104285
Normal_Group               528351
Normal_Cash_Withdrawal     305031
Normal_Cash_Deposits       223801
Normal_Periodical          210526
Normal_Plus_Mutual         155041
Normal_Mutual              125335
Normal_Foward               42031
Normal_single_large         20641
Name: count, dtype: int6

In [8]:
cat = pd.read_csv(file, usecols=['Payment_type', 'Sender_bank_location', 'Date'])

print("Payment types:")
print(cat['Payment_type'].value_counts())

print(f"\nSender locations: {cat['Sender_bank_location'].nunique()} distinct")
print(cat['Sender_bank_location'].value_counts().head(8))

print(f"\nDate range: {cat['Date'].min()} to {cat['Date'].max()}")
print(f"Distinct days: {cat['Date'].nunique()}")

Payment types:
Payment_type
Credit card        2012909
Debit card         2012103
Cheque             2011419
ACH                2008807
Cross-border        933931
Cash Withdrawal     300477
Cash Deposit        225206
Name: count, dtype: int64

Sender locations: 18 distinct
Sender_bank_location
UK             9183088
Turkey           20902
Switzerland      20503
Pakistan         20346
UAE              20081
Nigeria          20027
Spain            19391
Germany          19259
Name: count, dtype: int64

Date range: 2022-10-07 to 2023-08-23
Distinct days: 321


In [9]:
import numpy as np

cols = ['Time','Date','Sender_account','Receiver_account','Amount',
        'Payment_currency','Received_currency','Sender_bank_location',
        'Receiver_bank_location','Payment_type','Is_laundering','Laundering_type']

chunks_crime, chunks_clean = [], []
rng = np.random.default_rng(42)
keep_rate = 1_500_000 / 9_494_979

for chunk in pd.read_csv(file, usecols=cols, chunksize=500_000):
    chunks_crime.append(chunk[chunk['Is_laundering'] == 1])
    clean = chunk[chunk['Is_laundering'] == 0]
    mask = rng.random(len(clean)) < keep_rate
    chunks_clean.append(clean[mask])

df = pd.concat(chunks_crime + chunks_clean, ignore_index=True)
df = df.sort_values(['Date','Time']).reset_index(drop=True)

print(f"Sample rows:  {len(df):,}")
print(f"  criminal:   {df['Is_laundering'].sum():,}")
print(f"  clean:      {(df['Is_laundering']==0).sum():,}")
print(f"Memory:       {df.memory_usage(deep=True).sum()/1e9:.2f} GB")

Sample rows:  1,509,825
  criminal:   9,873
  clean:      1,499,952
Memory:       0.74 GB


In [10]:
out = '/content/drive/MyDrive/aml-project/sample.parquet'
df.to_parquet(out, index=False)

size = os.path.getsize(out) / 1e6
print(f"Saved: {size:.0f} MB")

Saved: 29 MB
